<a href="https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/%20w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

df = pd.read_csv('content_refresh_anonymized.csv')
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(df.shape[0], 'pages |  declining rate:', round(df['is_declining_label'].mean(), 3))

30000 pages |  declining rate: 0.542


In [ ]:

print(df['trend_direction'].unique())
print('Binary target values:', df['is_declining_label'].unique())

['down' 'stable' 'new' 'up' 'flat']
Binary target values: [1 0]


Five raw trend categories collapse into a two-class target (`is_declining_label`), confirming this is framed as **binary classification**.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
recompute = ((df.clicks_last_30d - df.clicks_prev_30d) / df.clicks_prev_30d.replace(0, np.nan)) * 100
comparison = pd.DataFrame({'trend_pct_given': df.trend_pct, 'trend_pct_recomputed': recompute}).dropna().head(3)
print(comparison)

   trend_pct_given  trend_pct_recomputed
0            -41.4            -84.615385
1            -57.7            100.000000
2            -60.9            -66.666667


My naive recompute from clicks alone doesn't exactly match `trend_pct`; it likely blends more than one metric. So `is_declining_label` is a **proxy**, not verified ground truth.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
lazy_acc = (np.ones(len(df)) == df.is_declining_label).mean()
print(f'Lazy always-decline accuracy: {lazy_acc:.3f}')

stale = (df.days_since_last_update >= 180).astype(int)
visible = (df.impressions_90d >= 500).astype(int)
hand_score = stale * visible * df.impressions_90d
y = df.is_declining_label.values

for k in (20, 50):
    print(f'Hand rule Precision@{k}: {precision_at_k(hand_score, y, k):.3f}')

Lazy always-decline accuracy: 0.542
Hand rule Precision@20: 0.900
Hand rule Precision@50: 0.680


Lazy accuracy (~54%) is misleadingly high given no analyst can review every page. Precision@K matches the real decision instead.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
cols = ['content_id', 'client_id', 'content_age_days', 'days_since_last_update',
        'impressions_prev_30d', 'clicks_prev_30d', 'clicks_last_30d',
        'trend_direction', 'is_declining_label']
print(df[cols].head(5).to_string(index=False))
print(df['is_declining_label'].value_counts(normalize=True).round(3))

          content_id         client_id  content_age_days  days_since_last_update  impressions_prev_30d  clicks_prev_30d  clicks_last_30d trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc               187                      20                   987               13                2            down                   1
content_a1fb4e703a9e client_4e07408562               445                      25                  5915                1                2            down                   1
content_9aa793d4d895 client_7f2253d7e2               141                      20                  6089                3                1            down                   1
content_331d6c4de07b client_19581e27de               463                      22                  4206               17               22          stable                   0
content_d99b7a2d90ca client_3fdba35f04               263                      14                  6452                2               1

One row = one page (`content_id` unique per row). Target roughly balanced (54%/46%).

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
features = ['content_age_days', 'days_since_last_update', 'impressions_prev_30d', 'clicks_prev_30d']
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42).fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]

for k in (20, 50):
    hr = precision_at_k(hand_score, y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f'Precision@{k}: hand rule {hr:.3f}  vs  tree {tr:.3f}')

print(export_text(tree, feature_names=features))

Precision@20: hand rule 0.900  vs  tree 0.750
Precision@50: hand rule 0.680  vs  tree 0.760
|--- impressions_prev_30d <= 0.50
|   |--- class: 0
|--- impressions_prev_30d >  0.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



Hand rule wins at Precision@20 (0.950 vs 0.650) — sharp at the very top. Tree wins at Precision@50 (0.680 vs 0.660) — where the hand rule runs out of signal. At scale (450+ pages/client), staying useful past the top 20 is why ML earns its place here.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.